# 04 Data Integration

For this notebook, it will combining the cleaned datasets from Yahoo Finance, Wikipedia, and FRED into a shared project dataset. The goal is to prepare on clean set of files that can then be used for feature engiener, modeling, portfolio optimization, and dashboard work.

# Imports

In [2]:
import pandas as pd
import numpy as np 
from pathlib import Path

# Paths

In [3]:
YFINANCE_PATH = Path('../data/processed/yfinance')
WIKIPEDIA_PATH = Path('../data/processed/wikipedia')
FRED_PATH = Path('../data/processed/fred')
INTEGRATED_PATH = Path('../data/processed/integrated')

In [4]:
if not INTEGRATED_PATH.exists():
    INTEGRATED_PATH.mkdir(parents=True, exist_ok=True)
else:
    print(f"{INTEGRATED_PATH} already exists.")

..\data\processed\integrated already exists.


# Load Cleaned Data

In [5]:
adj_close = pd.read_csv(
    YFINANCE_PATH / 'yfinance_adjusted_close_clean.csv',
    index_col=0,
    parse_dates=True
)

In [6]:
daily_returns = pd.read_csv(
    YFINANCE_PATH / 'yfinance_daily_returns_clean.csv',
    index_col=0,
    parse_dates=True
)

In [12]:
summary_stats = pd.read_csv(
    YFINANCE_PATH / "yfinance_processed_summary_stats.csv",
    index_col=0
)

In [13]:
risk_free_rate = pd.read_csv(
    FRED_PATH / "fred_risk_free_rate_processed.csv",
    index_col="date",
    parse_dates=True
)

In [14]:
cpi = pd.read_csv(
    FRED_PATH / "fred_cpi_processed.csv",
    index_col="date",
    parse_dates=True
)

In [15]:
sp500_metadata = pd.read_csv(
    WIKIPEDIA_PATH / "wikipedia_sp500_constituents_clean.csv"
)

# Inspect Data Shapes

In [16]:
print("Adjusted close:", adj_close.shape)
print("Daily returns:", daily_returns.shape)
print("Summary stats:", summary_stats.shape)
print("Risk-free rate:", risk_free_rate.shape)
print("CPI:", cpi.shape)
print("S&P 500 metadata:", sp500_metadata.shape)

Adjusted close: (2010, 21)
Daily returns: (2009, 21)
Summary stats: (21, 6)
Risk-free rate: (2010, 2)
CPI: (2892, 2)
S&P 500 metadata: (503, 3)


Now all cleaned datasets are loaded from their folders. The next step is to check whether the dates and ticker symbols are lined up correctly

# Check Date Alignment

In [17]:
print("Adjusted close date range:", adj_close.index.min(), "to", adj_close.index.max())
print("Daily returns date range:", daily_returns.index.min(), "to", daily_returns.index.max())
print("Risk-free rate date range:", risk_free_rate.index.min(), "to", risk_free_rate.index.max())
print("CPI date range:", cpi.index.min(), "to", cpi.index.max())

Adjusted close date range: 2018-01-02 00:00:00 to 2025-12-30 00:00:00
Daily returns date range: 2018-01-03 00:00:00 to 2025-12-30 00:00:00
Risk-free rate date range: 2018-01-02 00:00:00 to 2025-12-30 00:00:00
CPI date range: 2018-01-01 00:00:00 to 2025-12-01 00:00:00


In [21]:
print("Prices and risk-free rate same dates:", adj_close.index.equals(risk_free_rate.index))

# NOTE: Daily returns only need to be in the risk-free rate dates, not necessarily the same dates as prices
print("Daily returns dates are in risk-free rate:", daily_returns.index.isin(risk_free_rate.index).all())

Prices and risk-free rate same dates: True
Daily returns dates are in risk-free rate: True


The risk-free rate data is asligned to the same trading dates as the yfinance price data. Dialy returns start one day later because returns require a previous price to calculate the return. Therefore, the daily returns data will not have the same dates as the price data, but it should be a subset of the risk-free rate dates.

# Create Asset Metadata

In [22]:
portfolio_tickers = list(adj_close.columns)
print("Portfolio tickers:", portfolio_tickers)

Portfolio tickers: ['AAPL', 'AGG', 'AMZN', 'CAT', 'GLD', 'JPM', 'KO', 'LLY', 'LMT', 'MSFT', 'NEE', 'PG', 'QQQ', 'SPY', 'TLT', 'UNH', 'V', 'VNQ', 'VXUS', 'VZ', 'XOM']


In [23]:
asset_metadata = pd.DataFrame({
    "ticker": portfolio_tickers,
})

asset_metadata.head()

,ticker
0,AAPL
1,AGG
2,AMZN
3,CAT
4,GLD


# Merge Wikipedia Sector Metadata with Portfolio Tickers

In [ ]:
asset_metadata = asset_metadata.merge(
    sp500_metadata,
    on="ticker",
    how="left",
)

In [25]:
asset_metadata

,ticker,company_name,gics_sector
0,AAPL,Apple Inc.,Information Technology
1,AGG,NaN,NaN
2,AMZN,Amazon,Consumer Discretionary
3,CAT,Caterpillar Inc.,Industrials
4,GLD,NaN,NaN
5,JPM,JPMorgan Chase,Financials
6,KO,Coca-Cola Company (The),Consumer Staples
7,LLY,Lilly (Eli),Health Care
8,LMT,Lockheed Martin,Industrials
9,MSFT,Microsoft,Information Technology


# 9 Manually Fill in ETF / Non-S&P Metadata

In [26]:
# NOTE: I'm filling in data with what's stated on the ETF's website. This is a manual process and may not be perfectly accurate.
manual_metadata = {
    "AGG": {
        "company_name": "iShares Core U.S. Aggregate Bond ETF",
        "gics_sector": "Bonds"
    },
    "TLT": {
        "company_name": "iShares 20+ Year Treasury Bond ETF",
        "gics_sector": "Long-Term Treasury Bonds"
    },
    "GLD": {
        "company_name": "SPDR Gold Shares",
        "gics_sector": "Gold / Commodities"
    },
    "QQQ": {
        "company_name": "Invesco QQQ Trust",
        "gics_sector": "Growth / Nasdaq ETF"
    },
    "SPY": {
        "company_name": "SPDR S&P 500 ETF Trust",
        "gics_sector": "Benchmark / S&P 500 ETF"
    },
    "VNQ": {
        "company_name": "Vanguard Real Estate ETF",
        "gics_sector": "Real Estate ETF"
    },
    "VXUS": {
        "company_name": "Vanguard Total International Stock ETF",
        "gics_sector": "International Equity ETF"
    }
}

In [27]:
for ticker, metadata in manual_metadata.items():
    mask = asset_metadata["ticker"] == ticker
    
    asset_metadata.loc[mask, "company_name"] = asset_metadata.loc[mask, "company_name"].fillna(metadata["company_name"])
    asset_metadata.loc[mask, "gics_sector"] = asset_metadata.loc[mask, "gics_sector"].fillna(metadata["gics_sector"])

In [28]:
asset_metadata

,ticker,company_name,gics_sector
0,AAPL,Apple Inc.,Information Technology
1,AGG,iShares Core U.S. Aggregate Bond ETF,Bonds
2,AMZN,Amazon,Consumer Discretionary
3,CAT,Caterpillar Inc.,Industrials
4,GLD,SPDR Gold Shares,Gold / Commodities
5,JPM,JPMorgan Chase,Financials
6,KO,Coca-Cola Company (The),Consumer Staples
7,LLY,Lilly (Eli),Health Care
8,LMT,Lockheed Martin,Industrials
9,MSFT,Microsoft,Information Technology


# Double Check for Missing Metadata

In [29]:
asset_metadata.isna().sum()

ticker          0
company_name    0
gics_sector     0
dtype: int64

In [30]:
asset_metadata[asset_metadata.isna().any(axis=1)]

,ticker,company_name,gics_sector


Since some of the portfolios were ETFS and didn't have data they didn't appear in the S&p 500 company table. So those assets were manually labeled by data founded on Google so every ticker has a name and category for analysis. Then we validated that all tickers have a name and category by checking for missing values. The final asset metadata table is now ready to be used for analysis. 

# Add Asset Type To Data

In [31]:
ETF_TICKERS = ["AGG", "TLT", "GLD", "QQQ", "SPY", "VNQ", "VXUS"]

In [32]:
asset_metadata["asset_type"] = np.where(
    asset_metadata["ticker"].isin(ETF_TICKERS),
    "ETF",
    "Stock"
)

asset_metadata

,ticker,company_name,gics_sector,asset_type
0,AAPL,Apple Inc.,Information Technology,Stock
1,AGG,iShares Core U.S. Aggregate Bond ETF,Bonds,ETF
2,AMZN,Amazon,Consumer Discretionary,Stock
3,CAT,Caterpillar Inc.,Industrials,Stock
4,GLD,SPDR Gold Shares,Gold / Commodities,ETF
5,JPM,JPMorgan Chase,Financials,Stock
6,KO,Coca-Cola Company (The),Consumer Staples,Stock
7,LLY,Lilly (Eli),Health Care,Stock
8,LMT,Lockheed Martin,Industrials,Stock
9,MSFT,Microsoft,Information Technology,Stock
